<a href="https://colab.research.google.com/github/ashwin-inquisitor/Final_Year_project/blob/main/hfl_vanet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
from google.colab import drive
drive.mount('/content/drive')
package_path = '/content/packages'
drive_package_path = '/content/drive/My Drive/Packages'

# Check if the symbolic link already exists and remove it if it does
if os.path.exists(package_path) and os.path.islink(package_path):
    os.unlink(package_path)

# Create the symbolic link
os.symlink(drive_package_path, package_path)
sys.path.insert(0, package_path)

Mounted at /content/drive


In [15]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import traci
import random
from collections import deque

torch.manual_seed(0)
np.random.seed(0)

Model: simple MLP classifer

In [16]:
class MLP(nn.Module):
    def __init__(self, in_dim=20, hidden=64, out_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x):
        return self.net(x)

Synthetic non-IID dataset generator per client

In [17]:
def make_synthetic_client_data(n_samples=200, in_dim=20, n_classes=3, class_bias=None):
    X = np.random.randn(n_samples, in_dim).astype(np.float32)
    logits = np.random.randn(n_samples, n_classes)
    y = logits.argmax(axis=1)
    if class_bias is not None:
        mask = np.random.rand(n_samples) < 0.6
        y[mask] = class_bias
    return torch.tensor(X), torch.tensor(y)

Utilities: accuracy and loss

In [18]:
def compute_metrics(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        loss = nn.CrossEntropyLoss()(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean().item()
    return loss.item(), acc

def distance(p1, p2):
    return ((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)**0.5

Deep Gradient Compression: top-k masking + residuals

In [19]:
def topk_mask(tensor, k_ratio=0.1):
    flat = tensor.view(-1)
    k = max(1, int(k_ratio * flat.numel()))
    if k >= flat.numel():
        return torch.ones_like(tensor, dtype=torch.bool)
    _, idx = torch.topk(flat.abs(), k, sorted=False)
    mask = torch.zeros_like(flat, dtype=torch.bool)
    mask[idx] = True
    return mask.view_as(tensor)

def dgc_compress(grads, residuals, k_ratio=0.1, momentum=0.0):
    compressed, new_residuals = {}, {}
    for name in grads:
        g = grads[name]
        r = residuals.get(name, torch.zeros_like(g))
        g_hat = g + momentum * r
        mask = topk_mask(g_hat, k_ratio)
        comp = torch.where(mask, g_hat, torch.zeros_like(g_hat))
        res = torch.where(mask, torch.zeros_like(g_hat), g_hat)
        compressed[name] = comp
        new_residuals[name] = res
    return compressed, new_residuals

def apply_grad(model, grad_dict, lr=0.05):
    with torch.no_grad():
        for (name, p) in model.named_parameters():
            if name in grad_dict:
                p -= lr * grad_dict[name]

def average_grads(grad_list, weights=None):
    if weights is None:
        weights = [1.0 / len(grad_list)] * len(grad_list)
    out = {}
    for name in grad_list[0]:
        acc = 0.0
        for g, w in zip(grad_list, weights):
            acc = acc + w * g[name]
        out[name] = acc
    return out

Local training to produce raw gradients

In [20]:
def local_step_get_grad(model, X, y, lr=0.05, batch_size=64):
    model.train()
    opt = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    init_params = {n: p.clone().detach() for n, p in model.named_parameters()}

    idx = torch.randperm(X.size(0))
    Xb, yb = X[idx[:batch_size]], y[idx[:batch_size]]
    opt.zero_grad()
    logits = model(Xb)
    loss = criterion(logits, yb)
    loss.backward()
    opt.step()

    grads = {}
    with torch.no_grad():
        for (n, p) in model.named_parameters():
            grads[n] = (init_params[n] - p).detach() / lr
            p.copy_(init_params[n])
    return grads, loss.item()

EMA smoothing for convergence

In [21]:
class EMA:
  def __init__(self, beta=0.9):
    self.beta = beta
    self.value = None
  def update(self, x):
    if self.value is None:
      self.value = x
    else:
      self.value = self.beta * self.value + (1 - self.beta) * x
    return self.value

Entities: CM, CH, and EPC

In [22]:
class ClientCM:
    def __init__(self, cid, data):
        self.cid = cid
        self.X, self.y = data
        self.residuals = {}
    def compute_compressed_grad(self, model, k_ratio=0.1, momentum=0.0, lr=0.05):
        grads, loss = local_step_get_grad(model, self.X, self.y, lr=lr)
        comp, self.residuals = dgc_compress(grads, self.residuals, k_ratio=k_ratio, momentum=momentum)
        weight = len(self.y)
        return comp, weight, loss

class ClusterHead:
    def __init__(self, hid, clients):
        self.hid = hid
        self.clients = clients
        self.ema_loss = EMA(beta=0.9)
    def aggregate_from_clients(self, model, k_ratio=0.1, momentum=0.0, lr=0.05):
        grad_dicts, weights, losses = [], [], []
        for cm in self.clients:
            comp, w, loss = cm.compute_compressed_grad(model, k_ratio=k_ratio, momentum=momentum, lr=lr)
            grad_dicts.append(comp); weights.append(w); losses.append(loss)
        weights = (np.array(weights, dtype=np.float32) / np.sum(weights)).tolist()
        agg = average_grads(grad_dicts, weights)
        smoothed = self.ema_loss.update(float(np.mean(losses)))
        return agg, smoothed

class EPC:
    def __init__(self, chs):
        self.chs = chs
        self.ema_loss = EMA(beta=0.9)
    def aggregate_from_chs(self, grad_dicts, weights=None):
        return average_grads(grad_dicts, weights)

SUMO cluster builder

In [23]:
def get_vehicle_positions():
    vehicle_ids = traci.vehicle.getIDList()
    positions = {vid: traci.vehicle.getPosition(vid) for vid in vehicle_ids}
    return positions

def assign_clusters(vehicle_positions, ch_positions, radius=200):
    clusters = {ch: [] for ch in ch_positions}
    for vid, pos in vehicle_positions.items():
        if not ch_positions:
            continue
        nearest_ch = min(ch_positions, key=lambda ch: distance(pos, ch))
        if distance(pos, nearest_ch) <= radius:
            clusters[nearest_ch].append(vid)
    return clusters

def build_clusters_from_sumo(ch_positions, in_dim=20, n_classes=3):
    vehicle_positions = get_vehicle_positions()
    clusters_map = assign_clusters(vehicle_positions, ch_positions)
    clusters = []
    for ch_pos, members in clusters_map.items():
        clients = []
        for vid in members:
            data = make_synthetic_client_data(
                n_samples=200, in_dim=in_dim, n_classes=n_classes,
                class_bias=np.random.randint(0, n_classes)
            )
            clients.append(ClientCM(cid=vid, data=data))
        clusters.append(ClusterHead(hid=f"CH_{ch_pos}", clients=clients))
    return clusters

Communication constraints

In [24]:
PACKET_LOSS_PROB = 0.2   # probability to drop a CH->EPC update
MAX_DELAY = 2            # max delay (in rounds) for CH->EPC update
delayed_queue = deque()  # holds (due_round, grad_dict, weight)

Run simulation with SUMO

In [25]:
def run_with_sumo(rounds=20, ch_positions=[(0, 0), (500, 500)], lr=0.05, stop_eps=5e-4, stop_patience=4):
    model = MLP(in_dim=20, hidden=64, out_dim=3)
    X_val, y_val = make_synthetic_client_data(n_samples=500, in_dim=20, n_classes=3)
    epc = EPC([])

    last_smooth, patience = None, 0

    for r in range(rounds):
        traci.simulationStep()  # advance SUMO one tick
        chs = build_clusters_from_sumo(ch_positions)
        epc.chs = chs

        ch_grads, ch_weights = [], []

        for ch in chs:
            agg_grad, _ = ch.aggregate_from_clients(model)
            if not agg_grad:
                continue

            # Packet loss: drop some CH updates
            if random.random() < PACKET_LOSS_PROB:
                continue

            # Multi-hop delay: queue some updates to arrive in future rounds
            delay = random.randint(0, MAX_DELAY)
            weight = sum(len(cm.y) for cm in ch.clients)
            if delay > 0:
                delayed_queue.append((r + delay, agg_grad, weight))
            else:
                ch_grads.append(agg_grad)
                ch_weights.append(weight)

        # Process delayed updates due this round
        # Note: multiple items may be due; collect all
        remaining = deque()
        while delayed_queue:
            due_round, grad, weight = delayed_queue.popleft()
            if due_round == r:
                ch_grads.append(grad)
                ch_weights.append(weight)
            else:
                remaining.append((due_round, grad, weight))
        while remaining:
            delayed_queue.append(remaining.popleft())

        if not ch_grads:
            print(f"Round {r+1:02d} | No updates received (loss/delay/range)")
            continue

        ch_weights = (np.array(ch_weights, dtype=np.float32) / np.sum(ch_weights)).tolist()
        global_grad = epc.aggregate_from_chs(ch_grads, weights=ch_weights)
        apply_grad(model, global_grad, lr=lr)

        val_loss, val_acc = compute_metrics(model, X_val, y_val)
        smooth = epc.ema_loss.update(val_loss)
        print(f"Round {r+1:02d} | Val Loss: {val_loss:.4f} (EMA {smooth:.4f}) | Val Acc: {val_acc:.3f}")

        # Convergence check
        if last_smooth is not None and abs(smooth - last_smooth) < stop_eps:
            patience += 1
        else:
            patience = 0
        last_smooth = smooth
        if patience >= stop_patience:
            print(f"Converged at round {r+1}.")
            break

    print("SUMO-FL simulation complete with communication constraints.")